# Disclosure Gap Matrix

This notebook generates the publication-ready disclosure gap matrix. All company classifications are loaded from the `data/disclosure_coding.csv` table; no company-level disclosure values are hard-coded here.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import Wedge, FancyBboxPatch
from pathlib import Path

WORKBOOK_NAME = "disclosure_coding.csv"

def find_plot_project():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "plots_for_paper"]
    for candidate in candidates:
        if (candidate / "data" / WORKBOOK_NAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate data/{WORKBOOK_NAME}")

PROJECT_DIR = find_plot_project()
DATA_FILE = PROJECT_DIR / "data" / WORKBOOK_NAME
PLOTS_DIR = PROJECT_DIR / "plots"
plot_data = pd.read_csv(DATA_FILE).sort_values("Plot order")

# 2026-08-31: Facility-level and Region-resolved are no longer scored as two independent
# requirements -- they are merged into "Spatially attributable electricity" (Yes if either
# is Yes), and a new "Renewable accounting transparency" dimension replaces the freed slot.
field_map = {
    "DC-only MWh": "DC_total_MWh",
    "Spatially attributable electricity": "Spatially_attributable_MWh",
    "Hourly CFE": "Time_based_alignment",
    "Boundary": "Boundary_clarity",
    "Renewable accounting transparency": "Renewable_accounting_transparency",
}
status_names = {"Yes": "full", "Partial": "partial", "No": "none"}
companies = plot_data["Company"].tolist()
metrics = list(field_map.values())
gap_status = plot_data.set_index("Company")[list(field_map)].rename(columns=field_map)
unknown = set(gap_status.stack().dropna().unique()) - set(status_names)
if unknown:
    raise ValueError(f"Unknown disclosure status values in Plot_Data: {sorted(unknown)}")
disclosure_matrix = gap_status.replace(status_names).loc[companies, metrics].values.tolist()

matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 8
matplotlib.rcParams['axes.linewidth'] = 0.5
matplotlib.rcParams['xtick.major.width'] = 0.5
matplotlib.rcParams['ytick.major.width'] = 0.5

fig = plt.figure(figsize=(7.2, 4.2))
gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.2, 0.5], wspace=0.15)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax_companies = fig.add_subplot(gs[0, 2])

conditions = [
    "Data center electricity total\n(global MWh)",
    "Spatially attributable electricity\n(facility-level OR region-resolved)",
    "Time-based alignment\n(hourly CFE or equivalent)",
    "Boundary clarity\n(quantified inclusion/exclusion)",
    "Renewable-electricity accounting\ntransparency (Scope 2 method + instrument)"
]

ax1.set_xlim(-0.05, 1.0)
ax1.set_ylim(-0.3, 5.3)
ax1.axis('off')
ax1.text(0.5, 5.15, "Minimum Information for\nGrid-Relevant Interpretation",
         fontsize=8, ha='center', va='top', fontweight='bold')
for i, condition in enumerate(conditions):
    y_pos = 3.6 - i * 0.9
    if i % 2 == 0:
        ax1.add_patch(FancyBboxPatch(
            (-0.05, y_pos - 0.38), 1.05, 0.8, boxstyle="round,pad=0.02",
            facecolor='#F8F9FA', edgecolor='none', zorder=0
        ))
    ax1.text(0.0, y_pos, f"{i+1}.", fontsize=7, va='center', ha='left', fontweight='bold')
    ax1.text(0.12, y_pos, condition, fontsize=7, va='center', ha='left', wrap=True)

ax2.set_xlim(-0.3, 5.2)
ax2.set_ylim(-0.3, 5.3)
ax2.axis('off')
ax2.text(2.45, 5.15, "Current Disclosure Coverage", fontsize=8, ha='center', va='top', fontweight='bold')
ax2.plot([-0.3, -0.3], [-0.3, 5.3], color='#BDBDBD', linewidth=1.0, zorder=1)
ax2.add_patch(FancyBboxPatch(
    (-0.3, 4.15), 5.5, 0.7, boxstyle="round,pad=0.02",
    facecolor='#E8EAF6', edgecolor='#C5CAE9', linewidth=0.5, zorder=0
))
for j in range(5):
    ax2.text(j, 4.5, f"{j+1}", fontsize=7, va='center', ha='center', fontweight='bold')

def draw_circle(ax, x, y, status, size=80):
    radius = 0.11
    if status == 'full':
        ax.scatter(x, y, s=size, c='#4CAF50', marker='o', edgecolors='#388E3C', linewidths=0.8, zorder=3)
    elif status == 'partial':
        ax.scatter(x, y, s=size, c='white', marker='o', edgecolors='#0288D1', linewidths=0.8, zorder=3)
        ax.add_patch(Wedge((x, y), radius, 90, 270, facecolor='#4FC3F7', edgecolor='none', zorder=3))
    else:
        ax.scatter(x, y, s=size, c='white', marker='o', edgecolors='#BDBDBD', linewidths=0.8, zorder=3)

for i, statuses in enumerate(disclosure_matrix):
    y_pos = 3.6 - i * 0.9
    if i % 2 == 1:
        ax2.add_patch(FancyBboxPatch(
            (-0.3, y_pos - 0.38), 5.5, 0.8, boxstyle="round,pad=0.02",
            facecolor='#F8F9FA', edgecolor='none', zorder=0
        ))
    ax2.plot([-0.3, 5.2], [y_pos - 0.45, y_pos - 0.45], color='#E0E0E0', linewidth=0.5, zorder=1)
    for j, status in enumerate(statuses):
        draw_circle(ax2, j, y_pos, status)

ax_companies.set_xlim(-0.1, 0.5)
ax_companies.set_ylim(-0.3, 5.3)
ax_companies.axis('off')
for i, company in enumerate(companies):
    y_pos = 3.6 - i * 0.9
    if i % 2 == 1:
        ax_companies.add_patch(FancyBboxPatch(
            (-0.1, y_pos - 0.38), 0.6, 0.8, boxstyle="round,pad=0.02",
            facecolor='#F8F9FA', edgecolor='none', zorder=0
        ))
    ax_companies.text(0.0, y_pos, company, fontsize=7, va='center', ha='left')

legend_elements = [
    mpatches.Patch(facecolor='#4CAF50', edgecolor='#388E3C', label='Yes (quantitative, verifiable)'),
    mpatches.Patch(facecolor='#4FC3F7', edgecolor='#0288D1', label='Partial'),
    mpatches.Patch(facecolor='white', edgecolor='#BDBDBD', label='No')
]
ax2.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.15),
           fontsize=6, frameon=False, ncol=3, handlelength=1.2, handleheight=0.8, columnspacing=1.5)

fig.tight_layout(pad=0.5)
fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05)
fig.savefig(PLOTS_DIR / "disclosure_gap_matrix.pdf", format='pdf', dpi=600, facecolor='white')
fig.savefig(PLOTS_DIR / "disclosure_gap_matrix.png", format='png', dpi=600, facecolor='white')
plt.show()
print(f"Loaded {len(companies)} companies from {DATA_FILE.name} and saved disclosure_gap_matrix.pdf/.png")
